In [1]:
import sys 
sys.path.append('/host/d/Github')
import os
import torch
import numpy as np
import nibabel as nb
import random
import matplotlib.pyplot as plt

import torch
from torch.utils.data import Dataset

import CT_registration_diffusion.functions_collection as ff
import CT_registration_diffusion.Build_lists.Build_list as Build_list
import CT_registration_diffusion.Data_processing as Data_processing
import CT_registration_diffusion.Generator as Generator
import CT_registration_diffusion.model.model as model
import CT_registration_diffusion.model.train_engine as train_engine
import CT_registration_diffusion.model.predict_engine as predict_engine

/opt/conda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### define our trial name

In [2]:
trial_name = 'trial_1'

### step 0: define some pre-set parameters

In [3]:
# image
image_size = [224,224,96]
cutoff_range = [-200,250]

### step 1: define patient list

In [4]:
# change the excel path to your own path
patient_list_spreadsheet = os.path.join('/host/d/Data/CT_image/Patient_lists/ct_list.xlsx')
build_sheet =  Build_list.Build(patient_list_spreadsheet)

# define train
batch_list_train, dataset_id_list_train, case_id_list_train, image_folder_list_train = build_sheet.__build__(batch_list = [0,1,2])
# 先用一个case跑通代码
######## zhennong: 0:1, yuanqi: 1:2, leyu: 2:3, luxin: 3:4
# batch_list_train = batch_list_train[0:4]
# dataset_id_list_train = dataset_id_list_train[0:4]
# case_id_list_train = case_id_list_train[0:4]
# image_folder_list_train = image_folder_list_train[0:4]


# define validation
batch_list_val, dataset_id_list_val, case_id_list_val, image_folder_list_val = build_sheet.__build__(batch_list = [3])


# print一个路径来看看
print('train image folder:', image_folder_list_train[0])
print('Case id list',case_id_list_train,case_id_list_val)


train image folder: /host/d/Data/CT_image/DIR_LAB/Case1/cropped_image
Case id list ['Case1' 'Case2' 'Case1' 'Case3' 'Case4' 'Case2' 'Case5' 'Case6' 'Case7'
 'Case3'] ['Case8' 'Case4']


### step 2: define generator

In [5]:
# define training generator
only_use_tf0_as_moving = True # if set True, only use time frame 0 as moving image, otherwise randomly select moving time frame

generator_train = Generator.Dataset_4DCT(
    image_folder_list = image_folder_list_train,
    
    image_size = image_size, # target image size after center-crop
    cutoff_range = cutoff_range, # default cutoff range for CT images

    num_of_pairs_each_case = 5, # 在一个4DCT case中，随机选取多少对time frames（比如说我们选了time frame 0和time frame 2作为一对，time frame 1和time frame 3作为另一对，那么num_of_pairs_each_case就是2）
    preset_paired_tf = None, # 预设每个case中time frame的配对情况，比如说[(0,2),(1,3)]表示time frame 0和2作为一对，1和3作为一对。如果设置了这个参数，那么num_of_pairs_each_case就需要和这个list的长度一致。如果是None，那么每次从4DCT中随机选取num_of_pairs_each_case对time frames。
    only_use_tf0_as_moving = only_use_tf0_as_moving, 

    shuffle = True,

    augment = True, # whether to do data augmentation, in training set it to True
    augment_frequency = 0.5, )

# define validation generator
# 和training不同的是，我们在validation中要尽量保持数据的一致性，因此不进行shuffle和data augmentation。同时我们要设定preset_paired_tf，确保每次选取的time frame配对是一样的。
preset_paired_tf_val = [(0,1),(0,2),(0,4), (0,5),(0,6),(0,8)] # 预设validation中每个case的time frame配对情况
generator_val = Generator.Dataset_4DCT(
    image_folder_list = image_folder_list_val,
    image_size = image_size, 
    cutoff_range = cutoff_range, 

    num_of_pairs_each_case = len(preset_paired_tf_val), 
    preset_paired_tf = preset_paired_tf_val, 
    only_use_tf0_as_moving = only_use_tf0_as_moving,
    shuffle = False,
    augment = False, # whether to do data augmentation
    augment_frequency = 0.0, )


In [32]:
warped_root_stage1 = '/host/d/projects/registration/models/trial_1/results_stage1'

# stage 2 是 local refinement: 输入不再是原始 moving，而是 stage 1 的 coarse warped image
local_patch_size = [96, 96, 48]
local_patches_per_pair = 8

def case_key_from_folder(image_folder):
    folder = image_folder.rstrip('/\\')
    base = os.path.basename(folder)
    if base.lower() in ['cropped_image', 'cropped', 'images', 'image', 'img']:
        case_id = os.path.basename(os.path.dirname(folder))
        dataset_id = os.path.basename(os.path.dirname(os.path.dirname(folder)))
    else:
        case_id = base
        dataset_id = os.path.basename(os.path.dirname(folder))
    return f'{dataset_id}_{case_id}'

def build_center_patch_starts(image_folder_list, paired_tfs, patch_size):
    fixed_patch_starts = {}
    for image_folder in image_folder_list:
        case_id = case_key_from_folder(image_folder)
        timeframes = ff.find_all_target_files(['img*'], image_folder)
        for _, fixed_tf in paired_tfs:
            fixed_path = timeframes[fixed_tf]
            volume_shape = nb.load(fixed_path).shape
            start = []
            for dim, size in zip(volume_shape, patch_size):
                start.append(max((dim - size) // 2, 0))
            fixed_patch_starts[(case_id, fixed_tf)] = tuple(start)
    return fixed_patch_starts

generator_train = Generator.PatchCascadeDataset(
    image_folder_list=image_folder_list_train,
    warped_root=warped_root_stage1,
    patch_size=local_patch_size,
    patches_per_pair=local_patches_per_pair,
    image_size=image_size,
    cutoff_range=cutoff_range,
    num_of_pairs_each_case=4,
    preset_paired_tf=None,
    only_use_tf0_as_moving=True,
    shuffle=True,
    augment=True,
)

preset_paired_tf_val = [(0,1),(0,2),(0,4),(0,5),(0,6),(0,8)]
fixed_patch_starts_val = build_center_patch_starts(image_folder_list_val, preset_paired_tf_val, local_patch_size)
generator_val = Generator.PatchCascadeDataset(
    image_folder_list=image_folder_list_val,
    warped_root=warped_root_stage1,
    patch_size=local_patch_size,
    patches_per_pair=1,
    image_size=image_size,
    cutoff_range=cutoff_range,
    num_of_pairs_each_case=len(preset_paired_tf_val),
    preset_paired_tf=preset_paired_tf_val,
    only_use_tf0_as_moving=True,
    shuffle=False,
    augment=False,
    augment_frequency=0.0,
    fixed_patch_starts=fixed_patch_starts_val,
)


### step 3: model

In [33]:
# build model
our_model = model.Unet(
    problem_dimension = '3D',  # we are solving a 3D image registration problem
  
    input_channels = 2, # =1 如果只有一个4DCT time frame(比如只有time frame 0）作为模型输入；=2 如果有两个4DCT time frames（比如time frame 0和time frame 2作为moving和fixed image）作为模型输入
    out_channels = 3,  # =2 for 2D deformation field; =3 for 3D deformation field

    initial_dim = 4,  # default initial feature dimension after first conv layer
    dim_mults = (2,4,8,16),
    groups = 2,
    # None：不用attention，"False":linear attention, "True": full attention
    full_attn_paths = (None, None, None, None), # these are for downsampling and upsampling paths， 现在都是None因为要考虑GPU内存
    full_attn_bottleneck  = False, # this is for the middle bottleneck layer， 现在是None因为要考虑GPU内存
    act = 'ReLU',
)

in out is :  [(4, 8), (8, 16), (16, 32), (32, 64)]


### step 4: build trainer and start to train the model

In [34]:
# IMPORTANT: training parameters
regularization_weight = 0.01 # weight for deformation field smoothness regularization term， 这个是要通过测试来确定最佳取值
similarity_metric = 'MSE' # similarity metric for image similarity loss, can be 'MSE' or 'NCC'

train_batch_size = 1 # training batch size,  如果GPU内存不够，可以把这个值设成1
accumulation_steps = 4 # gradient accumulation steps to simulate larger batch size， 如果GPU内存不够，可以把这个值设成更小

# training schedule
total_epochs = 300
save_models_every = 10 # save model every N epoc-------------hs，训练样本越少这个数字越大，样本越大这个数字可以越小（通常我设成1-5之间，如果只有一个case我会设成20-50）
validation_every = save_models_every # validate every N epochs, should be same as save_models_every
# where to save your model weights? Change this path to your own path
current_stage = 2
results_folder = os.path.join('/host/d/projects/registration/models', trial_name, f'models_stage{current_stage}')
ff.make_folder([os.path.basename(results_folder), results_folder])

trainer = train_engine.Trainer(
        our_model,
        generator_train,
        generator_val,
        train_batch_size = train_batch_size,
        accum_iter= accumulation_steps,

        regularization_weight = regularization_weight,
        similarity_metric = similarity_metric,
        
        train_num_steps = total_epochs,
        results_folder = results_folder,
       
        train_lr_decay_every = 200, 
        save_models_every = save_models_every,
        validation_every = validation_every,)

In [35]:
### do we have pre-trained model?
#pretrained_model = os.path.join(results_folder, 'model-70.pt')

# what is the start epoch?
start_epoch = 0 # if no pre-trained model, start from epoch 0, else start from the loaded epoch


In [37]:
print('len(generator_train) =', len(generator_train))
print('len(generator_val) =', len(generator_val))

len(generator_train) = 320
len(generator_val) = 12


In [38]:
# # # start training!!!
trainer.train(pre_trained_model = None, start_step = start_epoch)
# #如果跑不动（GPU内存不足），
# 1.可以尝试减小model里initial_dim的值，比如改成4或者2
# 2.可以尝试减小train_batch_size
# 3.可以尝试减小accumulation_steps

  0%|          | 0/300 [00:00<?, ?it/s]

training epoch:  1
learning rate:  0.0001


average loss: 0.0074, average similarity loss: 0.0073, average regularization loss: 0.0100:   0%|          | 1/300 [03:24<16:57:03, 204.09s/it]

training epoch:  2
learning rate:  0.0001


average loss: 0.0067, average similarity loss: 0.0066, average regularization loss: 0.0018:   1%|          | 2/300 [06:47<16:51:54, 203.74s/it]

...

validation at step:  210


average loss: 0.0033, average similarity loss: 0.0030, average regularization loss: 0.0258:  70%|███████   | 210/300 [11:11:36<4:48:23, 192.27s/it]

validation loss: 0.0046, validation similarity loss: 0.0042, validation regularization loss: 0.0343
training epoch:  211
learning rate:  9.5e-05


average loss: 0.0033, average similarity loss: 0.0030, average regularization loss: 0.0258:  70%|███████   | 210/300 [11:12:00<4:48:00, 192.00s/it]


KeyboardInterrupt: 

### step 5: build predictor and use trained model to predict

In [39]:
# define save folder
save_folder = os.path.join('/host/d/projects/registration/models', trial_name, 'results')
ff.make_folder([os.path.basename(save_folder), save_folder])

In [40]:
# change the excel path to your own path
patient_list_spreadsheet = os.path.join('/host/d/Data/CT_image/Patient_lists/ct_list.xlsx')
build_sheet =  Build_list.Build(patient_list_spreadsheet)

# define test (作为展示我们先用train的case来跑一下)
batch_list_tst, dataset_id_list_tst, case_id_list_tst, image_folder_list_tst = build_sheet.__build__(batch_list = [0,1,2,3,4])
# 先用一个case跑通代码
######## zhennong: 0:1, yuanqi: 1:2, leyu: 2:3, luxin: 3:4
# batch_list_tst = batch_list_tst[0:4]
# dataset_id_list_tst = dataset_id_list_tst[0:4]
# case_id_list_tst = case_id_list_tst[0:4]
# image_folder_list_tst = image_folder_list_tst[0:4]

### step 5.2 define the pre-trained model (the epoch that has the best validation loss)

In [41]:
epoch = 210
trained_model_file = os.path.join('/host/d/projects/registration/models', trial_name, 'models_stage2', 'model-' + str(epoch) + '.pt')

### step 5.3 do the prediction

In [43]:

# for i in range(0,case_id_list_tst.shape[0]):
    
#     # find out how many time frames in this 4DCT
#     image_folder = image_folder_list_tst[i]
#     timeframes = ff.sort_timeframe(ff.find_all_target_files(['img*'], image_folder),2,'_')
#     print('case id:', case_id_list_tst[i], 'has', len(timeframes), 'time frames.')

#     # save folder for this case
#     save_folder_case = os.path.join(save_folder, case_id_list_tst[i], 'epoch_' + str(epoch))
#     ff.make_folder([os.path.basename(os.path.basename(save_folder_case)), os.path.basename(save_folder_case), save_folder_case])
    
#     ### get the deformation fields for each time frame using the first time frame as moving image
#     for tf in range(1,len(timeframes)):# range(1, len(timeframes)):
#         moving_tf = 0
#         fixed_tf = tf

#         # define prediction generator
#         generator_pred = Generator.Dataset_4DCT(
#             image_folder_list = [image_folder_list_tst[i]],
#             image_size = image_size, 
#             cutoff_range = cutoff_range, 
#             only_use_tf0_as_moving=True,

#             num_of_pairs_each_case = 1, 
#             preset_paired_tf = [(moving_tf, fixed_tf)], 
#         )

#         # define predictor
#         predictor = predict_engine.Predictor(
#             our_model,
#             generator_pred,
#             batch_size = 1,
#         )

        
#         # predict MVF
#         pred_MVF, pred_MVF_numpy, warped_moving_image_numpy = predictor.predict_MVF_and_apply(trained_model_filename = trained_model_file)
#         print('predicted MVF_numpy shape:', pred_MVF_numpy.shape)
#         print('warped moving image shape:', warped_moving_image_numpy.shape)

#         # save truth
#         moving_image_file = timeframes[moving_tf]
#         fixed_image_file = timeframes[fixed_tf]
#         moving_image_nii = nb.load(moving_image_file)
#         fixed_image_nii = nb.load(fixed_image_file)
#         affine = moving_image_nii.affine

#         nb.save(nb.Nifti1Image(moving_image_nii.get_fdata(), affine), os.path.join(save_folder_case, 'gt_tf' + str(moving_tf) + '.nii.gz'))
#         nb.save(nb.Nifti1Image(fixed_image_nii.get_fdata(), affine), os.path.join(save_folder_case, 'gt_tf' + str(fixed_tf) + '.nii.gz'))
#         # save warped moving image
#         nb.save(nb.Nifti1Image(warped_moving_image_numpy, affine), os.path.join(save_folder_case, 'warped_tf' + str(fixed_tf) + '.nii.gz'))
#         # save predicted MVF
#         nb.save(nb.Nifti1Image(pred_MVF_numpy[0,...], affine), os.path.join(save_folder_case, 'pred_MVF_tf' + str(fixed_tf) + '_x.nii.gz'))
#         nb.save(nb.Nifti1Image(pred_MVF_numpy[1,...], affine), os.path.join(save_folder_case, 'pred_MVF_tf' + str(fixed_tf) + '_y.nii.gz'))
#         nb.save(nb.Nifti1Image(pred_MVF_numpy[2,...], affine), os.path.join(save_folder_case, 'pred_MVF_tf' + str(fixed_tf) + '_z.nii.gz'))





import glob

save_root = '/host/d/projects/registration/models/trial_1'
results_stage1 = save_root + '/results_stage1'
results_stage2 = save_root + '/results_stage2'
results_stage3 = save_root + '/results_stage3'

trained_model_stage1 = '/host/d/projects/registration/models/trial_1/models_stage1/model-60.pt'
trained_model_stage2 = '/host/d/projects/registration/models/trial_1/models_stage2/model-210.pt'
trained_model_stage3 = '/host/d/projects/registration/models/trial_1/models_stage3/model-70.pt'

# stage 2/3 的推理同样要走 patch predictor，不能再沿用 whole-volume Predictor
local_patch_size = [96, 96, 48]
local_patch_stride = [48, 48, 24]

def resolve_case_id(image_folder):
    return case_key_from_folder(image_folder)

def resolve_warped_path(warped_root, image_folder, fixed_tf):
    case_id = resolve_case_id(image_folder)
    candidates = [
        os.path.join(warped_root, case_id, f'warped_tf{fixed_tf}.nii.gz'),
        os.path.join(warped_root, f'warped_tf{fixed_tf}.nii.gz'),
    ]
    epoch_hits = glob.glob(os.path.join(warped_root, case_id, 'epoch_*', f'warped_tf{fixed_tf}.nii.gz'))
    if epoch_hits:
        candidates.append(sorted(epoch_hits)[-1])
    for candidate in candidates:
        if os.path.exists(candidate):
            return candidate
    raise FileNotFoundError(f'Cannot find warped image for case={case_id}, tf={fixed_tf}')

def preprocess_volume(volume):
    volume = Data_processing.cutoff_intensity(volume, cutoff_range[0], cutoff_range[1])
    return Data_processing.normalize_image(
        volume,
        normalize_factor='equatoin',
        image_max=cutoff_range[1],
        image_min=cutoff_range[0],
        invert=False,
        final_max=1,
        final_min=0,
    )

def denormalize_volume(volume):
    return Data_processing.normalize_image(
        volume,
        normalize_factor='equatoin',
        image_max=cutoff_range[1],
        image_min=cutoff_range[0],
        invert=True,
        final_max=1,
        final_min=0,
    )

def run_prediction(stage, warped_root_in, save_folder_out, trained_model_file, patch_size=None, patch_stride=None):
    patch_size = patch_size or local_patch_size
    patch_stride = patch_stride or local_patch_stride
    pred_mvf_numpy = None
    warped_moving_image_numpy = None

    for i in range(0, case_id_list_tst.shape[0]):
        image_folder = image_folder_list_tst[i]
        case_id = resolve_case_id(image_folder)
        timeframes = ff.sort_timeframe(ff.find_all_target_files(['img*'], image_folder), 2, '_')
        print('case id:', case_id, 'has', len(timeframes), 'time frames.')

        save_folder_case = os.path.join(save_folder_out, case_id, 'epoch_' + str(epoch))
        ff.make_folder([os.path.basename(save_folder_case), save_folder_case])

        for tf in range(1, len(timeframes)):
            moving_tf = 0
            fixed_tf = tf
            moving_image_file = timeframes[moving_tf]
            fixed_image_file = timeframes[fixed_tf]
            moving_image_nii = nb.load(moving_image_file)
            fixed_image_nii = nb.load(fixed_image_file)
            affine = moving_image_nii.affine

            if stage == 1:
                generator_pred = Generator.Dataset_4DCT(
                    image_folder_list=[image_folder],
                    image_size=image_size,
                    cutoff_range=cutoff_range,
                    num_of_pairs_each_case=1,
                    preset_paired_tf=[(moving_tf, fixed_tf)],
                    only_use_tf0_as_moving=True,
                    stage=1,
                    warped_root=None,
                    shuffle=False,
                    augment=False,
                    augment_frequency=0.0,
                )
                predictor = predict_engine.Predictor(our_model, generator_pred, batch_size=1)
                _, pred_mvf_numpy, warped_moving_image_numpy = predictor.predict_MVF_and_apply(
                    trained_model_filename=trained_model_file
                )
            else:
                coarse_path = resolve_warped_path(warped_root_in, image_folder, fixed_tf)
                coarse_volume = preprocess_volume(nb.load(coarse_path).get_fdata())
                fixed_volume = preprocess_volume(fixed_image_nii.get_fdata())
                predictor = predict_engine.PatchPredictor(
                    our_model,
                    patch_size=patch_size,
                    stride=patch_stride,
                    device='cuda',
                )
                pred_mvf_numpy, warped_moving_image_numpy_norm, patch_metadata = predictor.refine_volume(
                    coarse_warped_image=coarse_volume,
                    fixed_image=fixed_volume,
                    trained_model_filename=trained_model_file,
                )
                warped_moving_image_numpy = denormalize_volume(warped_moving_image_numpy_norm)
                print('num local patches:', len(patch_metadata))

            print('predicted MVF_numpy shape:', pred_mvf_numpy.shape)
            print('warped moving image shape:', warped_moving_image_numpy.shape)

            nb.save(nb.Nifti1Image(moving_image_nii.get_fdata(), affine), os.path.join(save_folder_case, 'gt_tf' + str(moving_tf) + '.nii.gz'))
            nb.save(nb.Nifti1Image(fixed_image_nii.get_fdata(), affine), os.path.join(save_folder_case, 'gt_tf' + str(fixed_tf) + '.nii.gz'))
            nb.save(nb.Nifti1Image(warped_moving_image_numpy, affine), os.path.join(save_folder_case, 'warped_tf' + str(fixed_tf) + '.nii.gz'))
            nb.save(nb.Nifti1Image(pred_mvf_numpy[0,...], affine), os.path.join(save_folder_case, 'pred_MVF_tf' + str(fixed_tf) + '_x.nii.gz'))
            nb.save(nb.Nifti1Image(pred_mvf_numpy[1,...], affine), os.path.join(save_folder_case, 'pred_MVF_tf' + str(fixed_tf) + '_y.nii.gz'))
            nb.save(nb.Nifti1Image(pred_mvf_numpy[2,...], affine), os.path.join(save_folder_case, 'pred_MVF_tf' + str(fixed_tf) + '_z.nii.gz'))

    return pred_mvf_numpy, warped_moving_image_numpy


In [24]:
pred_MVF_numpy, warped_moving_image_numpy = run_prediction(
    stage=1,
    warped_root_in=None,
    save_folder_out=results_stage1,
    trained_model_file=trained_model_stage1
)
print('Stage 1 global MVF range:', pred_MVF_numpy.min(), pred_MVF_numpy.max())

case id: DIR_LAB_Case1 has 10 time frames.


  0%|          | 0/1 [00:00<?, ?it/s]

STAGE 1 moving_path: /host/d/Data/CT_image/DIR_LAB/Case1/cropped_image/img_0.nii.gz fixed_path: /host/d/Data/CT_image/DIR_LAB/Case1/cropped_image/img_1.nii.gz


100%|██████████| 1/1 [00:02<00:00,  2.62s/it]


predicted MVF_numpy shape: (3, 224, 224, 96)
warped moving image shape: (224, 224, 96)


...

predicted MVF_numpy shape: (3, 224, 224, 96)
warped moving image shape: (224, 224, 96)


  0%|          | 0/1 [00:00<?, ?it/s]

STAGE 1 moving_path: /host/d/Data/CT_image/Popi/Case6/cropped_image/img_0.nii.gz fixed_path: /host/d/Data/CT_image/Popi/Case6/cropped_image/img_9.nii.gz


100%|██████████| 1/1 [00:01<00:00,  1.17s/it]


predicted MVF_numpy shape: (3, 224, 224, 96)
warped moving image shape: (224, 224, 96)
Stage 1 global MVF range: -1.8925992 6.6874123


In [44]:
pred_MVF_numpy, warped_moving_image_numpy = run_prediction(
    stage=2,
    warped_root_in=results_stage1,
    save_folder_out=results_stage2,
    trained_model_file=trained_model_stage2
)
print('Stage 2 local residual MVF range:', pred_MVF_numpy.min(), pred_MVF_numpy.max())

case id: DIR_LAB_Case1 has 10 time frames.
num local patches: 48
predicted MVF_numpy shape: (3, 224, 224, 96)
warped moving image shape: (224, 224, 96)
num local patches: 48
predicted MVF_numpy shape: (3, 224, 224, 96)
warped moving image shape: (224, 224, 96)
num local patches: 48
predicted MVF_numpy shape: (3, 224, 224, 96)
warped moving image shape: (224, 224, 96)
num local patches: 48
predicted MVF_numpy shape: (3, 224, 224, 96)
warped moving image shape: (224, 224, 96)
num local patches: 48
predicted MVF_numpy shape: (3, 224, 224, 96)
warped moving image shape: (224, 224, 96)
num local patches: 48
predicted MVF_numpy shape: (3, 224, 224, 96)
warped moving image shape: (224, 224, 96)
num local patches: 48
predicted MVF_numpy shape: (3, 224, 224, 96)
warped moving image shape: (224, 224, 96)
num local patches: 48
predicted MVF_numpy shape: (3, 224, 224, 96)
warped moving image shape: (224, 224, 96)
num local patches: 48
predicted MVF_numpy shape: (3, 224, 224, 96)
warped moving imag

In [45]:
# # print('data range:', np.min(pred_MVF_numpy), np.max(pred_MVF_numpy))

# pred_MVF_numpy, warped_moving_image_numpy = run_prediction( 
#     stage=1,
#     warped_root_in=None,
#     save_folder_out=results_stage1,
#     trained_model_file=trained_model_stage1)
# print(pred_MVF_numpy.min(), pred_MVF_numpy.max())

# pred_MVF_numpy, warped_moving_image_numpy = run_prediction( 
#     stage=2,
#     warped_root_in=results_stage1,
#     save_folder_out=results_stage2,
#     trained_model_file=trained_model_stage2)
# print(pred_MVF_numpy.min(), pred_MVF_numpy.max())

# pred_MVF_numpy, warped_moving_image_numpy = run_prediction( 
#     stage=3,
#     warped_root_in=results_stage2,
#     save_folder_out=results_stage3,
#     trained_model_file=trained_model_stage3)
# print(pred_MVF_numpy.min(), pred_MVF_numpy.max())


In [49]:
import nibabel as nb
import numpy as np
import torch
import CT_registration_diffusion.model.loss as my_loss

# 统一评估配置：只保留两个阶段
eval_case_id = 'DIR_LAB_Case1'  # 改成你实际评估的 dataset_case，例如 DIR_LAB_Case5 / Popi_Case5
eval_tf = 3
eval_stage1_epoch = 'epoch_60'
eval_stage2_epoch = 'epoch_210'
eval_stage_roots = {
    'stage1': (results_stage1, eval_stage1_epoch),
    'stage2': (results_stage2, eval_stage2_epoch),
}

def load_and_normalize_nii(path):
    img = nb.load(path).get_fdata()
    return Data_processing.normalize_image(
        img,
        normalize_factor='equation',
        image_max=cutoff_range[1],
        image_min=cutoff_range[0],
        invert=False,
        final_max=1,
        final_min=0,
    )

def build_eval_paths(case_id, tf_index):
    paths = {}
    for stage_name, (stage_root, epoch_name) in eval_stage_roots.items():
        case_root = os.path.join(stage_root, case_id, epoch_name)
        paths[stage_name] = {
            'fixed': os.path.join(case_root, f'gt_tf{tf_index}.nii.gz'),
            'warped': os.path.join(case_root, f'warped_tf{tf_index}.nii.gz'),
        }
    return paths

eval_paths = build_eval_paths(eval_case_id, eval_tf)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
ncc_loss = my_loss.NCCLoss()

for stage_name, stage_paths in eval_paths.items():
    fixed_image = load_and_normalize_nii(stage_paths['fixed'])
    warped_image = load_and_normalize_nii(stage_paths['warped'])
    fixed_torch = torch.from_numpy(fixed_image[np.newaxis, np.newaxis, ...]).float().to(device)
    warped_torch = torch.from_numpy(warped_image[np.newaxis, np.newaxis, ...]).float().to(device)
    value = ncc_loss(warped_torch, fixed_torch)
    print(f'NCC for {stage_name}: {value.item():.6f}')


NCC for stage1: -0.220688
NCC for stage2: -0.240449


### Calculate MAE

In [50]:
def compute_mae(img1, img2):
    return np.mean(np.abs(img1 - img2))

for stage_name, stage_paths in eval_paths.items():
    fixed_image = load_and_normalize_nii(stage_paths['fixed'])
    warped_image = load_and_normalize_nii(stage_paths['warped'])
    mae_value = compute_mae(warped_image, fixed_image)
    print(f'MAE for {stage_name}: {mae_value:.6f}')


MAE for stage1: 1.450486
MAE for stage2: 1.446350


In [3]:
from skimage.metrics import structural_similarity as ssim
import os
import numpy as np

def compute_ssim(img1, img2, win_size=7):
    return ssim(img1, img2, data_range=1.0, win_size=win_size)

for stage_name, stage_paths in eval_paths.items():
    fixed_image = load_and_normalize_nii(stage_paths['fixed'])
    warped_image = load_and_normalize_nii(stage_paths['warped'])
    ssim_value = compute_ssim(warped_image, fixed_image)
    print(f'SSIM for {stage_name}: {ssim_value:.6f}')


NameError: name 'eval_paths' is not defined

In [4]:
from pathlib import Path
import shutil
import tempfile

lobe_names = [
    'lung_lower_lobe_left',
    'lung_lower_lobe_right',
    'lung_middle_lobe_right',
    'lung_upper_lobe_left',
    'lung_upper_lobe_right',
]

mask_root_candidates = [
    Path(r'D:\\Masks'),
    Path('/host/d/Masks'),
    Path('/mnt/d/Masks'),
]
mask_root = next((path for path in mask_root_candidates if path.exists()), None)
if mask_root is None:
    raise FileNotFoundError('Cannot find the Masks folder. Please check D:/Masks or update mask_root_candidates.')

fixed_mask_dir = mask_root / 'fixed_masks'
warped_mask_dir = mask_root / 'warped_masks'

def find_mask_path(mask_dir, lobe_name):
    exact_path = mask_dir / f'{lobe_name}.nii.gz'
    if exact_path.exists():
        return exact_path
    fuzzy_matches = sorted(mask_dir.glob(f'{lobe_name}*.nii.gz'))
    if fuzzy_matches:
        return fuzzy_matches[0]
    raise FileNotFoundError(f'Cannot find mask for {lobe_name} in {mask_dir}')

def load_binary_mask(mask_path):
    mask_path = Path(mask_path)
    with open(mask_path, 'rb') as f:
        file_header = f.read(2)

    is_gzip = file_header == b'\x1f\x8b'

    if is_gzip or mask_path.suffix != '.gz':
        mask_data = nb.load(str(mask_path)).get_fdata()
    else:
        # Some TotalSegmentator outputs here are plain .nii files saved with a .nii.gz suffix.
        with tempfile.TemporaryDirectory() as tmp_dir:
            tmp_path = Path(tmp_dir) / mask_path.name.replace('.nii.gz', '.nii')
            shutil.copyfile(mask_path, tmp_path)
            mask_data = nb.load(str(tmp_path)).get_fdata()

    return mask_data > 0

def compute_dice(mask_a, mask_b):
    if mask_a.shape != mask_b.shape:
        raise ValueError(f'Shape mismatch: {mask_a.shape} vs {mask_b.shape}')
    intersection = np.logical_and(mask_a, mask_b).sum()
    total = mask_a.sum() + mask_b.sum()
    if total == 0:
        return 1.0
    return 2.0 * intersection / total

dice_scores = {}
fixed_masks = []
warped_masks = []

for lobe_name in lobe_names:
    fixed_path = find_mask_path(fixed_mask_dir, lobe_name)
    warped_path = find_mask_path(warped_mask_dir, lobe_name)

    fixed_mask = load_binary_mask(fixed_path)
    warped_mask = load_binary_mask(warped_path)

    dice_scores[lobe_name] = compute_dice(fixed_mask, warped_mask)
    fixed_masks.append(fixed_mask)
    warped_masks.append(warped_mask)

print('Dice per lobe:')
for lobe_name, dice_value in dice_scores.items():
    print(f'{lobe_name}: {dice_value:.4f}')

mean_dice = np.mean(list(dice_scores.values()))
overall_fixed_mask = np.logical_or.reduce(fixed_masks)
overall_warped_mask = np.logical_or.reduce(warped_masks)
overall_dice = compute_dice(overall_fixed_mask, overall_warped_mask)

print(f'Mean Dice across 5 lobes: {mean_dice:.4f}')
print(f'Overall lung Dice: {overall_dice:.4f}')


Dice per lobe:
lung_lower_lobe_left: 0.4855
lung_lower_lobe_right: 0.6788
lung_middle_lobe_right: 0.2057
lung_upper_lobe_left: 0.4540
lung_upper_lobe_right: 0.6762
Mean Dice across 5 lobes: 0.5000
Overall lung Dice: 0.7788
